# Run the same feature-prep job through Glue ETL and a Wrangler-export Processing Job and prove which one wins on cost and time

The churn-modeling lead has been arguing with the analytics team for two sprints over whether Glue Spark ETL or SageMaker Data Wrangler should handle the feature-prep step that feeds Feature Store. The data engineers want Glue because they already write Spark and they know the per-DPU-minute math. The analytics team wants Data Wrangler because they want to build feature pipelines in a browser without writing Spark. You are the engineer the lead asked to settle it: run the same job in both tools against the same 50 MB customer CSV, capture each job's DPU-seconds and wall-clock time, then write up the call. Production volume is 5 GB per day, the analyst headcount is three, and Data Wrangler's free-tier window closes in a month. The math is the artifact; the recommendation is the deliverable.

Five artifacts ship in this lab:

- An S3 bucket holding a deterministic 50 MB customer churn CSV under `input/customers.csv`.
- An IAM role for the Glue Python shell job (S3 RW plus the Glue service trust).
- A Glue Python shell job that null-fills, one-hot encodes, and standard-scales the CSV, writing Parquet to `output/glue/`.
- An IAM role for the Wrangler-equivalent SageMaker Processing job (S3 RW plus the SageMaker service trust).
- A SageMaker Processing job running the sklearn framework container against the same input, writing Parquet to `output/wrangler/`.

**Time.** Plan for about 60 minutes hands-on. The Glue Python shell run takes about 3 to 5 minutes. The SageMaker Processing job takes about 4 to 8 minutes (container pull plus the same null-fill, one-hot, scale work). IAM propagation adds 5 to 10 seconds after each role is created.

**Cost.** Glue Python shell at the smallest DPU setting (0.0625) for about 5 minutes lands at roughly $0.002. The SageMaker Processing job on `ml.m5.large` at $0.115 per hour for about 5 minutes lands at roughly $0.01. S3 PUT and GET at this scale are inside the always-free tier. Session range: about $0.00 to $0.05 if cleanup runs on the first try. Data Wrangler's first 25 hours per month are free for 2 months on a new account; this lab uses the sklearn-container fallback so the cost story is the same whether or not the free-tier window is open.

This lab maps to AWS MLA-C01 Domain 1 (Data Preparation for ML, 28%). The hands-on construction is two pipelines against the same input. The cognitive work is the comparison reasoning: which tool wins given the team's headcount and the daily volume the pipeline will actually see.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12. The bucket name appends the 12-digit account id and lands
# at the 63-character S3 limit; the IAM role names sit at or just under
# the 64-character IAM limit. No abbreviations needed.

import atexit
import csv
import getpass
import io
import json
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "glue-etl-and-data-wrangler-for-feature-prep"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

GLUE_ROLE_NAME = f"labrun-{LAB_ID}-glue-role"
WRANGLER_ROLE_NAME = f"labrun-{LAB_ID}-wrangler-role"
GLUE_JOB_NAME = f"labrun-{LAB_ID}-job"
GLUE_INLINE_POLICY = "labrun-mla-lab02-glue-s3-policy"
WRANGLER_INLINE_POLICY = "labrun-mla-lab02-wrangler-s3-policy"

INPUT_KEY = "input/customers.csv"
GLUE_SCRIPT_KEY = "scripts/feature_prep.py"
WRANGLER_SCRIPT_KEY = "scripts/prep_script.py"
GLUE_OUTPUT_PREFIX = "output/glue/"
WRANGLER_OUTPUT_PREFIX = "output/wrangler/"

# Account-derived state is resolved after STS returns.
BUCKET_NAME = None
ACCOUNT_ID = None
GLUE_ROLE_ARN = None
WRANGLER_ROLE_ARN = None
WRANGLER_JOB_NAME = None

# Measurement bag populated by Tasks 2 and 3 and read by Task 5.
glue_dpu_seconds = 0.0
glue_wall_clock_seconds = 0.0
wrangler_wall_clock_seconds = 0.0
wrangler_instance_type = "ml.m5.large"

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names. This cell must succeed before the manifest
# cell creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 1 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
GLUE_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{GLUE_ROLE_NAME}"
WRANGLER_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{WRANGLER_ROLE_NAME}"
print(f"Bucket: {BUCKET_NAME}")
print(f"Glue role ARN: {GLUE_ROLE_ARN}")
print(f"Wrangler-equivalent role ARN: {WRANGLER_ROLE_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan.
# Manifest is module-level and in reverse-creation order. labrun-checks
# 0.6.0 dispatch supports glue_job, iam_role (which deletes inline
# policies first), and s3_bucket (which empties the bucket's first page
# of up to 1000 objects then deletes it). The Wrangler-equivalent
# Processing job is not in the manifest because completed Processing jobs
# cannot be deleted via API and bill zero at rest; the cleanup cell calls
# stop_processing_job manually if the job is still running.
#
# Nothing in this lab is hourly-billed at rest, so the manifest carries
# no critical flag. Glue Python shell bills per DPU-second only while
# running; SageMaker Processing bills per second only while running.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="glue_job",
        id=GLUE_JOB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {GLUE_JOB_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=GLUE_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {GLUE_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=WRANGLER_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {WRANGLER_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Run the cleanup")
    print("cell at the bottom of this notebook first, or remove them manually with")
    print("the AWS CLI commands the cleanup cell prints, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the input bucket and seed it with a 50 MB customer churn CSV

Both pipelines need the same input. The 50 MB target matters: it is large enough to make the per-DPU-minute math real (Glue Python shell at the 0.0625 DPU minimum still has to scan and parse the whole file) but small enough that the SageMaker Processing job's container pull dominates the wall clock, which is part of the point of the comparison.

Build it in this order:

1. Create the bucket `labrun-glue-etl-and-data-wrangler-for-feature-prep-{your-account-id}` and tag it `labrun:lab-slug=glue-etl-and-data-wrangler-for-feature-prep` so cleanup and the orphan scan can find it.
2. Generate a deterministic CSV with `numpy.random.default_rng(seed=42)` and 500,000 rows. Columns: `customer_id`, `tenure`, `monthly_charges`, `total_charges`, `contract`, `payment_method`, `churn`. Around 5 percent of `tenure` and 5 percent of `total_charges` values must be empty (the null-fill step in Tasks 2 and 3 is the lesson; without missing values it has nothing to do).
3. Upload the CSV to `s3://{bucket}/input/customers.csv`.

Determinism matters here. The same seed across both pipelines means Task 4's row-count comparison is meaningful: any drift is an actual bug, not a coin flip.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the bucket, generate the deterministic 50 MB customer CSV,
# upload it to s3://{bucket}/input/customers.csv.

import numpy as np

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the bucket using s3.create_bucket(Bucket=BUCKET_NAME).
# us-east-1 rejects LocationConstraint in create_bucket; other regions
# require CreateBucketConfiguration={"LocationConstraint": REGION}.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# Deterministic CSV generation. 500,000 rows lands at roughly 50 MB once
# the columns below are written; the exact byte count depends on the
# random distribution of total_charges digits. The checkpoint accepts
# anything between 40 and 60 MB.
rng = np.random.default_rng(seed=42)
n_rows = 500_000
contracts = np.array(["Month-to-month", "One year", "Two year"])
payments = np.array(["Electronic check", "Mailed check", "Bank transfer", "Credit card"])

customer_id = np.arange(n_rows)
tenure = rng.integers(0, 72, size=n_rows).astype(float)
monthly_charges = np.round(rng.uniform(18.25, 118.75, size=n_rows), 2)
total_charges = np.round(monthly_charges * np.maximum(tenure, 1), 2)
contract = contracts[rng.integers(0, 3, size=n_rows)]
payment_method = payments[rng.integers(0, 4, size=n_rows)]
churn = rng.integers(0, 2, size=n_rows)

# Inject ~5 percent nulls in tenure and total_charges. The lesson in
# Tasks 2 and 3 is median imputation; without missing values it has
# nothing to do.
null_mask_t = rng.random(n_rows) < 0.05
null_mask_c = rng.random(n_rows) < 0.05
tenure[null_mask_t] = np.nan
total_charges[null_mask_c] = np.nan

buf = io.StringIO()
writer = csv.writer(buf)
writer.writerow([
    "customer_id", "tenure", "monthly_charges", "total_charges",
    "contract", "payment_method", "churn",
])
for i in range(n_rows):
    t_val = "" if np.isnan(tenure[i]) else f"{tenure[i]:.0f}"
    tc_val = "" if np.isnan(total_charges[i]) else f"{total_charges[i]:.2f}"
    writer.writerow([
        int(customer_id[i]), t_val, f"{monthly_charges[i]:.2f}", tc_val,
        contract[i], payment_method[i], int(churn[i]),
    ])
csv_bytes = buf.getvalue().encode("utf-8")
print(f"Generated CSV in memory: {len(csv_bytes) / (1024 * 1024):.1f} MB, {n_rows:,} rows.")

# YOUR CODE: Upload the CSV using s3.put_object() with Bucket=BUCKET_NAME,
# Key=INPUT_KEY, and Body=csv_bytes.

print(f"Uploaded s3://{BUCKET_NAME}/{INPUT_KEY}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: input bucket holds a 50 MB CSV (40-60 MB tolerance) at
# input/customers.csv with the expected column header.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            head = s3_client.head_object(Bucket=BUCKET_NAME, Key=INPUT_KEY)
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str in ("404", "NoSuchKey", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"No object at s3://{BUCKET_NAME}/{INPUT_KEY}. Upload the "
                        f"customer CSV before running the pipelines."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        size_bytes = head.get("ContentLength", 0)
        size_mb = size_bytes / (1024 * 1024)
        if not (40 <= size_mb <= 60):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"input/customers.csv is {size_mb:.1f} MB; the lab expects 40-60 MB. "
                    f"Re-run Task 1 with the deterministic seed and 500,000 rows."
                ),
            )

        try:
            head_bytes = s3_client.get_object(
                Bucket=BUCKET_NAME, Key=INPUT_KEY, Range="bytes=0-4095",
            )["Body"].read()
        except ClientError as e:
            return CheckpointResult(status="error", error_reason=str(e))

        first_line = head_bytes.split(b"\n", 1)[0].decode("utf-8", errors="replace")
        expected_cols = {
            "customer_id", "tenure", "monthly_charges", "total_charges",
            "contract", "payment_method", "churn",
        }
        header_cols = {c.strip() for c in first_line.split(",")}
        missing = expected_cols - header_cols
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"CSV header is missing columns {sorted(missing)!r}. Found: "
                    f"{sorted(header_cols)!r}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Two API calls are missing in the cell: bucket creation and the CSV upload. The checkpoint message tells you which of size, key, or header is wrong. Read the failure before peeking further.

</details>

<details><summary>Hint 2 (direction)</summary>

In `us-east-1` use plain `s3.create_bucket(Bucket=BUCKET_NAME)` with no `CreateBucketConfiguration`. The `csv_bytes` variable already holds the encoded CSV; `s3.put_object(Bucket=..., Key=..., Body=csv_bytes)` is the entire upload.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
s3.create_bucket(Bucket=BUCKET_NAME)
s3.put_object(Bucket=BUCKET_NAME, Key=INPUT_KEY, Body=csv_bytes)
```

</details>

**Wallet check.** One bucket, one 50 MB PUT, the always-free S3 control-plane calls. S3 PUT at this scale sits inside the always-free tier; storage of 50 MB for an hour is fractions of a thousandth of a cent. Damage so far: about $0.00. Your morning coffee is leading by infinity.

## Task 2: Build the Glue Python shell pipeline and capture its DPU-seconds

Architecture A. A Glue Python shell job that null-fills numerics with the median, one-hot encodes the two categorical columns with `drop_first=True` (the ML-stable default), standard-scales the numeric columns, and writes Parquet to `output/glue/`. Glue Python shell at the 0.0625 DPU minimum is the lab's cheap path; Spark would charge a 10-minute minimum at 2 DPU.

Build it in this order:

1. Create the IAM role `labrun-glue-etl-and-data-wrangler-for-feature-prep-glue-role` with a trust policy for `glue.amazonaws.com` and an inline policy granting S3 RW on the lab bucket.
2. Upload the `feature_prep.py` script (baked into the cell as a multi-line string) to `s3://{bucket}/scripts/feature_prep.py`.
3. Create the Glue job with `Command={"Name": "pythonshell", "ScriptLocation": ...}`, `MaxCapacity=0.0625`, and the lab tag. Start a run and pass `--BUCKET_NAME` and `--INPUT_KEY` as job arguments.
4. Poll `get_job_run` until `JobRunState=SUCCEEDED`. After completion, poll up to 60 seconds for `DPUSeconds` to populate (Glue lags the bill reading up to 30 seconds after the job ends).

The captured `DPUSeconds` plus wall-clock time go into the comparison metric in Task 5.

In [ ]:
# NBVAL_SKIP
# Task 2: Glue role, feature_prep.py upload, Glue Python shell job,
# job run, poll until SUCCEEDED, capture DPU-seconds.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

glue_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "glue.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
glue_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
        },
        {
            "Effect": "Allow",
            "Action": ["s3:ListBucket"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "*",
        },
    ],
}

try:
    # YOUR CODE: Create the Glue role using iam.create_role() with
    # RoleName=GLUE_ROLE_NAME,
    # AssumeRolePolicyDocument=json.dumps(glue_trust_policy),
    # Description="labrun mla lab 02 glue role",
    # Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Created Glue role: {GLUE_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Glue role {GLUE_ROLE_NAME} already exists, reusing.")
    else:
        raise

# YOUR CODE: Attach the inline policy using iam.put_role_policy() with
# RoleName=GLUE_ROLE_NAME, PolicyName=GLUE_INLINE_POLICY,
# PolicyDocument=json.dumps(glue_inline_policy).

print(f"Attached inline policy {GLUE_INLINE_POLICY} to {GLUE_ROLE_NAME}")

# IAM propagation. Glue start_job_run rejects roles that have not
# propagated. Give it 10 seconds.
print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)

# The feature_prep.py script. Runs inside the Glue Python shell
# container, which ships with pandas, numpy, scikit-learn, and pyarrow
# preinstalled. Job arguments arrive on sys.argv as --BUCKET_NAME,
# --INPUT_KEY, and --OUTPUT_KEY; awsglue.utils.getResolvedOptions parses them.
FEATURE_PREP_SCRIPT = '''
import sys
import boto3
import pandas as pd
from io import BytesIO
from awsglue.utils import getResolvedOptions

args = getResolvedOptions(sys.argv, ["BUCKET_NAME", "INPUT_KEY", "OUTPUT_KEY"])
bucket = args["BUCKET_NAME"]
input_key = args["INPUT_KEY"]
output_key = args["OUTPUT_KEY"]

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=bucket, Key=input_key)
df = pd.read_csv(BytesIO(obj["Body"].read()))

# Median imputation on numeric columns.
for col in ["tenure", "total_charges"]:
    df[col] = df[col].fillna(df[col].median())

# One-hot encode categoricals with drop_first=True (ML-stable default).
df = pd.get_dummies(df, columns=["contract", "payment_method"], drop_first=True)

# Standard-scale numeric features.
for col in ["tenure", "monthly_charges", "total_charges"]:
    mu = df[col].mean()
    sigma = df[col].std()
    if sigma > 0:
        df[col] = (df[col] - mu) / sigma

buf = BytesIO()
df.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=bucket, Key=output_key, Body=buf.getvalue())
print("Wrote", len(df), "rows and", len(df.columns), "columns to", output_key)
'''

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=GLUE_SCRIPT_KEY,
    Body=FEATURE_PREP_SCRIPT.encode("utf-8"),
)
print(f"Uploaded script: s3://{BUCKET_NAME}/{GLUE_SCRIPT_KEY}")

try:
    # YOUR CODE: Create the Glue job using glue.create_job() with
    # Name=GLUE_JOB_NAME, Role=GLUE_ROLE_NAME,
    # Command={"Name": "pythonshell", "ScriptLocation":
    #     f"s3://{BUCKET_NAME}/{GLUE_SCRIPT_KEY}", "PythonVersion": "3.9"},
    # GlueVersion="3.0", MaxCapacity=0.0625, Timeout=20,
    # DefaultArguments={"--BUCKET_NAME": BUCKET_NAME,
    #                   "--INPUT_KEY": INPUT_KEY,
    #                   "--OUTPUT_KEY": f"{GLUE_OUTPUT_PREFIX}part-00000.parquet"},
    # Tags={LAB_TAG_KEY: LAB_TAG_VALUE}.
    print(f"Created Glue job: {GLUE_JOB_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue job {GLUE_JOB_NAME} already exists, reusing.")
    else:
        raise

# YOUR CODE: Start the run using glue.start_job_run(JobName=GLUE_JOB_NAME).
# Assign the response to run_resp.

run_id = run_resp["JobRunId"]
print(f"Started Glue job run: {run_id}")
print("Asking the Glue Python shell to roll up its sleeves, this takes 3 to 5 minutes...")

glue_start_wall = time.time()
final_run = None
elapsed = 0
while elapsed < 600:
    run_desc = glue.get_job_run(JobName=GLUE_JOB_NAME, RunId=run_id)["JobRun"]
    state = run_desc["JobRunState"]
    if state in ("SUCCEEDED", "FAILED", "ERROR", "TIMEOUT", "STOPPED"):
        final_run = run_desc
        break
    time.sleep(15)
    elapsed += 15

if final_run is None:
    print("Glue run did not finish within 10 minutes. Stop and investigate.")
    raise SystemExit(1)
if final_run["JobRunState"] != "SUCCEEDED":
    print(f"Glue run state: {final_run['JobRunState']}.")
    print(f"Error: {final_run.get('ErrorMessage', '(no message)')}")
    raise SystemExit(1)

glue_wall_clock_seconds = time.time() - glue_start_wall
print(f"Glue run finished in {glue_wall_clock_seconds:.1f} seconds wall clock.")

# DPUSeconds lags up to 30 seconds after completion. Poll for 60 seconds.
print("Asking Glue nicely for its DPU receipt...")
deadline = time.time() + 60
while time.time() < deadline:
    run_desc = glue.get_job_run(JobName=GLUE_JOB_NAME, RunId=run_id)["JobRun"]
    candidate = run_desc.get("DPUSeconds") or 0.0
    if candidate > 0:
        glue_dpu_seconds = float(candidate)
        break
    time.sleep(5)
else:
    # Fall back to ExecutionTime * MaxCapacity if DPUSeconds never landed.
    glue_dpu_seconds = float(run_desc.get("ExecutionTime", 0)) * 0.0625

print(f"Glue DPU-seconds captured: {glue_dpu_seconds:.2f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Glue job exists, last run state is SUCCEEDED, output
# prefix contains at least one .parquet object, and DPU-seconds captured
# in this session is positive.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job_desc = glue_client.get_job(JobName=GLUE_JOB_NAME)["Job"]
        except ClientError as e:
            if e.response["Error"]["Code"] in (
                "EntityNotFoundException", "ResourceNotFoundException",
            ):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Glue job {GLUE_JOB_NAME} not found.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job_desc.get("Role") not in (GLUE_ROLE_NAME, GLUE_ROLE_ARN):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Glue job {GLUE_JOB_NAME} is not using role {GLUE_ROLE_NAME}. "
                    f"Found role: {job_desc.get('Role')}."
                ),
            )

        runs = glue_client.get_job_runs(JobName=GLUE_JOB_NAME, MaxResults=1).get("JobRuns", [])
        if not runs:
            return CheckpointResult(
                status="fail",
                error_reason=f"Glue job {GLUE_JOB_NAME} has no run history. Start a run.",
            )
        last_run = runs[0]
        if last_run.get("JobRunState") != "SUCCEEDED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Last Glue run state is {last_run.get('JobRunState')!r}. The job has "
                    f"to reach SUCCEEDED before this checkpoint passes. Error: "
                    f"{last_run.get('ErrorMessage', '(no message)')!r}"
                ),
            )

        out = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=GLUE_OUTPUT_PREFIX)
        parquet_keys = [
            o["Key"] for o in out.get("Contents", []) if o["Key"].endswith(".parquet")
        ]
        if not parquet_keys:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No .parquet object found under s3://{BUCKET_NAME}/{GLUE_OUTPUT_PREFIX}. "
                    f"The Glue job ran but the script did not write its output here."
                ),
            )

        if glue_dpu_seconds <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "glue_dpu_seconds is zero. The Glue run succeeded but DPUSeconds "
                    "did not land in time. Re-run the task cell; the poll waits up to "
                    "60 seconds for DPU-seconds to populate."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Four API calls are missing in the cell: create the role, attach the inline policy, create the Glue job, and start the run. The checkpoint message names which of those is wrong. If you see a `JobRunState` other than `SUCCEEDED`, read the Glue logs in the CloudWatch console for the real error.

</details>

<details><summary>Hint 2 (direction)</summary>

`iam.create_role` takes Tags in the EC2 shape `[{"Key": ..., "Value": ...}]`. `glue.create_job` takes the same shape but as a dict: `Tags={LAB_TAG_KEY: LAB_TAG_VALUE}`. The Python shell job's `Command.Name` must be the literal string `"pythonshell"`. `MaxCapacity=0.0625` is the smallest billable DPU; the next step up is 1.0. `glue.start_job_run` returns `{"JobRunId": ...}`; capture the whole response so `run_resp["JobRunId"]` works.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
iam.create_role(
    RoleName=GLUE_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(glue_trust_policy),
    Description="labrun mla lab 02 glue role",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=GLUE_ROLE_NAME,
    PolicyName=GLUE_INLINE_POLICY,
    PolicyDocument=json.dumps(glue_inline_policy),
)

glue.create_job(
    Name=GLUE_JOB_NAME,
    Role=GLUE_ROLE_NAME,
    Command={
        "Name": "pythonshell",
        "ScriptLocation": f"s3://{BUCKET_NAME}/{GLUE_SCRIPT_KEY}",
        "PythonVersion": "3.9",
    },
    GlueVersion="3.0",
    MaxCapacity=0.0625,
    Timeout=20,
    DefaultArguments={
        "--BUCKET_NAME": BUCKET_NAME,
        "--INPUT_KEY": INPUT_KEY,
        "--OUTPUT_KEY": f"{GLUE_OUTPUT_PREFIX}part-00000.parquet",
    },
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

run_resp = glue.start_job_run(JobName=GLUE_JOB_NAME)
```

</details>

**Wallet check.** Glue Python shell at 0.0625 DPU for about 5 minutes is roughly 0.0625 DPU times 5/60 hours times $0.44 per DPU-hour, which is about $0.0023. The 1-minute billing minimum is the floor. IAM `CreateRole`, `PutRolePolicy`, and one Glue `CreateJob` plus `StartJobRun` are free of charge. Damage so far: fractions of a cent. Your coffee just laughed.

## Task 3: Build the Wrangler-equivalent pipeline as a SageMaker Processing job

Architecture B. A SageMaker Processing job running the sklearn framework container against a parameterized Python script that performs the same null-fill, one-hot, and standard-scale operations against the same input. The brief calls this the "Wrangler-export pattern": when a Data Wrangler flow outgrows the GUI, the team exports it as a Python script and runs it as a Processing Job. That export-to-script path is the production-ready surface of Data Wrangler; the GUI itself is for the analyst-friendly draft.

A pragmatic deviation from the brief, called out here so you know. Constructing valid Data Wrangler `.flow` JSON from scratch is brittle and the Data Wrangler container's input contract is undocumented enough that a notebook authored against it would break on the next Wrangler release. The sklearn container is stable, runs the same operations, and bills against the same instance-type story, so the comparison reasoning in Task 5 is unchanged.

Build it in this order:

1. Create the IAM role `labrun-glue-etl-and-data-wrangler-for-feature-prep-wrangler-role` with a trust policy for `sagemaker.amazonaws.com` and an inline policy granting S3 RW on the lab bucket.
2. Upload `prep_script.py` (baked into the cell as a multi-line string) to `s3://{bucket}/scripts/prep_script.py`.
3. Resolve the sklearn image URI via `sagemaker.image_uris.retrieve("sklearn", region="us-east-1", version="1.2-1")`.
4. Call `sagemaker.create_processing_job` with `ml.m5.large` (one instance), pass the bucket name and keys as `ProcessingInputs`/`ProcessingOutputs`, kick it off, and poll `describe_processing_job` until `ProcessingJobStatus=Completed`.

`ml.m5.large` is the smaller, cheaper SageMaker Processing instance ($0.115/hour). The brief mentions `ml.m5.4xlarge` for Data Wrangler ($0.922/hour); with the sklearn-container fallback, `ml.m5.large` is sufficient and keeps the lab's cost in pennies.

In [ ]:
# NBVAL_SKIP
# Task 3: Wrangler-equivalent role, prep_script.py upload, SageMaker
# Processing job on the sklearn container, poll until Completed, capture
# wall-clock seconds.

# image_uris.retrieve is the safe way to get the regional sklearn URI.
from sagemaker import image_uris

sagemaker_client = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

wrangler_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
wrangler_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
        },
        {
            "Effect": "Allow",
            "Action": ["s3:ListBucket"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "*",
        },
    ],
}

try:
    # YOUR CODE: Create the Wrangler-equivalent role using iam.create_role()
    # with RoleName=WRANGLER_ROLE_NAME,
    # AssumeRolePolicyDocument=json.dumps(wrangler_trust_policy),
    # Description="labrun mla lab 02 wrangler role",
    # Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Created Wrangler-equivalent role: {WRANGLER_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {WRANGLER_ROLE_NAME} already exists, reusing.")
    else:
        raise

# YOUR CODE: Attach the inline policy using iam.put_role_policy() with
# RoleName=WRANGLER_ROLE_NAME, PolicyName=WRANGLER_INLINE_POLICY,
# PolicyDocument=json.dumps(wrangler_inline_policy).

print(f"Attached inline policy {WRANGLER_INLINE_POLICY} to {WRANGLER_ROLE_NAME}")

# SageMaker propagation. CreateProcessingJob rejects roles that have not
# propagated. Give it 10 seconds.
print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)

# The prep_script.py. Same three operations as feature_prep.py but
# parameterized for the Processing job's input/output mount points
# (the Processing container surfaces inputs at /opt/ml/processing/input
# and writes outputs to /opt/ml/processing/output).
PREP_SCRIPT = '''
import os
import pandas as pd

INPUT_DIR = "/opt/ml/processing/input"
OUTPUT_DIR = "/opt/ml/processing/output"

csv_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")]
input_path = os.path.join(INPUT_DIR, csv_files[0])
df = pd.read_csv(input_path)

for col in ["tenure", "total_charges"]:
    df[col] = df[col].fillna(df[col].median())

df = pd.get_dummies(df, columns=["contract", "payment_method"], drop_first=True)

for col in ["tenure", "monthly_charges", "total_charges"]:
    mu = df[col].mean()
    sigma = df[col].std()
    if sigma > 0:
        df[col] = (df[col] - mu) / sigma

output_path = os.path.join(OUTPUT_DIR, "part-00000.parquet")
df.to_parquet(output_path, index=False)
print("Wrote", len(df), "rows and", len(df.columns), "columns to", output_path)
'''

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=WRANGLER_SCRIPT_KEY,
    Body=PREP_SCRIPT.encode("utf-8"),
)
print(f"Uploaded script: s3://{BUCKET_NAME}/{WRANGLER_SCRIPT_KEY}")

# YOUR CODE: Resolve the sklearn image URI using
# image_uris.retrieve(framework="sklearn", region=REGION, version="1.2-1").
# Assign it to SKLEARN_IMAGE_URI.

print(f"sklearn image URI: {SKLEARN_IMAGE_URI}")

WRANGLER_JOB_NAME = f"labrun-mla-lab02-wrangler-{int(time.time())}"
print(f"Processing job name: {WRANGLER_JOB_NAME}")

# YOUR CODE: Create the processing job using
# sagemaker_client.create_processing_job() with
# ProcessingJobName=WRANGLER_JOB_NAME,
# RoleArn=WRANGLER_ROLE_ARN,
# ProcessingResources={"ClusterConfig": {"InstanceCount": 1,
#     "InstanceType": wrangler_instance_type, "VolumeSizeInGB": 30}},
# AppSpecification={"ImageUri": SKLEARN_IMAGE_URI,
#     "ContainerEntrypoint": ["python3", "/opt/ml/processing/code/prep_script.py"]},
# ProcessingInputs=[
#     {"InputName": "input-csv",
#      "S3Input": {"S3Uri": f"s3://{BUCKET_NAME}/{INPUT_KEY}",
#                  "LocalPath": "/opt/ml/processing/input",
#                  "S3DataType": "S3Prefix", "S3InputMode": "File"}},
#     {"InputName": "prep-script",
#      "S3Input": {"S3Uri": f"s3://{BUCKET_NAME}/{WRANGLER_SCRIPT_KEY}",
#                  "LocalPath": "/opt/ml/processing/code",
#                  "S3DataType": "S3Prefix", "S3InputMode": "File"}},
# ],
# ProcessingOutputConfig={"Outputs": [
#     {"OutputName": "prepared",
#      "S3Output": {"S3Uri": f"s3://{BUCKET_NAME}/{WRANGLER_OUTPUT_PREFIX}",
#                   "LocalPath": "/opt/ml/processing/output",
#                   "S3UploadMode": "EndOfJob"}}]},
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].

print(f"Started Processing job: {WRANGLER_JOB_NAME}")
print("Tiptoeing through the sklearn container while it boots, this takes 4 to 8 minutes...")

final_proc = None
elapsed = 0
while elapsed < 900:
    proc_desc = sagemaker_client.describe_processing_job(
        ProcessingJobName=WRANGLER_JOB_NAME,
    )
    status = proc_desc["ProcessingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        final_proc = proc_desc
        break
    time.sleep(20)
    elapsed += 20

if final_proc is None:
    print("Processing job did not finish within 15 minutes. Stop and investigate.")
    raise SystemExit(1)
if final_proc["ProcessingJobStatus"] != "Completed":
    print(f"Processing job status: {final_proc['ProcessingJobStatus']}.")
    print(f"Failure reason: {final_proc.get('FailureReason', '(no reason)')}")
    raise SystemExit(1)

start_t = final_proc["ProcessingStartTime"]
end_t = final_proc["ProcessingEndTime"]
wrangler_wall_clock_seconds = (end_t - start_t).total_seconds()
print(f"Processing job finished in {wrangler_wall_clock_seconds:.1f} seconds (ProcessingStartTime to ProcessingEndTime).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Processing job exists with status Completed, output
# prefix holds at least one .parquet object, image URI is the sklearn
# pattern for the region, wrangler_wall_clock_seconds is positive.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        if not WRANGLER_JOB_NAME:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "WRANGLER_JOB_NAME is empty. Re-run the Task 3 cell so the "
                    "Processing job name gets assigned."
                ),
            )

        try:
            proc = sm_client.describe_processing_job(ProcessingJobName=WRANGLER_JOB_NAME)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not describe Processing job {WRANGLER_JOB_NAME}: "
                    f"{e.response['Error']['Code']}: {e.response['Error']['Message']}."
                ),
            )

        status_str = proc["ProcessingJobStatus"]
        if status_str != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Processing job status is {status_str!r}. The job has to reach "
                    f"Completed before this checkpoint passes. FailureReason: "
                    f"{proc.get('FailureReason', '(no reason)')!r}"
                ),
            )

        image_uri = proc.get("AppSpecification", {}).get("ImageUri", "")
        if "scikit-learn" not in image_uri and "sklearn" not in image_uri:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Processing job image URI {image_uri!r} does not look like the sklearn "
                    f"framework container. Use sagemaker.image_uris.retrieve(\"sklearn\", "
                    f"region=\"{REGION}\", version=\"1.2-1\")."
                ),
            )

        out = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=WRANGLER_OUTPUT_PREFIX)
        parquet_keys = [
            o["Key"] for o in out.get("Contents", []) if o["Key"].endswith(".parquet")
        ]
        if not parquet_keys:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"No .parquet object found under s3://{BUCKET_NAME}/"
                    f"{WRANGLER_OUTPUT_PREFIX}. The Processing job ran but the script "
                    f"did not write its output here."
                ),
            )

        if wrangler_wall_clock_seconds <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "wrangler_wall_clock_seconds is zero. Re-run the Task 3 cell so the "
                    "poll captures ProcessingStartTime and ProcessingEndTime."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Four API calls are missing in the cell: create the role, attach the inline policy, resolve the sklearn image URI, and create the Processing job. The checkpoint message names which of those failed. If the Processing job ran but the checkpoint still fails on image URI, you may have hardcoded a different region's URI; use `image_uris.retrieve` instead.

</details>

<details><summary>Hint 2 (direction)</summary>

The SageMaker Processing job needs three S3 wirings: input CSV mounted at `/opt/ml/processing/input`, script mounted at `/opt/ml/processing/code`, output written from `/opt/ml/processing/output`. The script's entrypoint must point at the mounted script path: `["python3", "/opt/ml/processing/code/prep_script.py"]`. The role's trust policy needs `Principal.Service: sagemaker.amazonaws.com`. `image_uris.retrieve(framework="sklearn", region=REGION, version="1.2-1")` returns the right container.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
iam.create_role(
    RoleName=WRANGLER_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(wrangler_trust_policy),
    Description="labrun mla lab 02 wrangler role",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=WRANGLER_ROLE_NAME,
    PolicyName=WRANGLER_INLINE_POLICY,
    PolicyDocument=json.dumps(wrangler_inline_policy),
)

SKLEARN_IMAGE_URI = image_uris.retrieve(
    framework="sklearn", region=REGION, version="1.2-1",
)

sagemaker_client.create_processing_job(
    ProcessingJobName=WRANGLER_JOB_NAME,
    RoleArn=WRANGLER_ROLE_ARN,
    ProcessingResources={
        "ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": wrangler_instance_type,
            "VolumeSizeInGB": 30,
        }
    },
    AppSpecification={
        "ImageUri": SKLEARN_IMAGE_URI,
        "ContainerEntrypoint": ["python3", "/opt/ml/processing/code/prep_script.py"],
    },
    ProcessingInputs=[
        {"InputName": "input-csv",
         "S3Input": {"S3Uri": f"s3://{BUCKET_NAME}/{INPUT_KEY}",
                     "LocalPath": "/opt/ml/processing/input",
                     "S3DataType": "S3Prefix", "S3InputMode": "File"}},
        {"InputName": "prep-script",
         "S3Input": {"S3Uri": f"s3://{BUCKET_NAME}/{WRANGLER_SCRIPT_KEY}",
                     "LocalPath": "/opt/ml/processing/code",
                     "S3DataType": "S3Prefix", "S3InputMode": "File"}},
    ],
    ProcessingOutputConfig={
        "Outputs": [
            {"OutputName": "prepared",
             "S3Output": {"S3Uri": f"s3://{BUCKET_NAME}/{WRANGLER_OUTPUT_PREFIX}",
                          "LocalPath": "/opt/ml/processing/output",
                          "S3UploadMode": "EndOfJob"}}
        ]
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** SageMaker Processing on `ml.m5.large` at $0.115 per hour for about 5 minutes is roughly 0.115 times 5/60, which is about $0.0096. IAM `CreateRole`, `PutRolePolicy`, and SageMaker `CreateProcessingJob` are free of charge. Damage so far: about $0.01 plus the fractions-of-a-cent Glue run. Your coffee is still ahead by a long way.

## Task 4: Verify both outputs have the same schema and row count

Both pipelines should produce identical schemas and row counts. If they do not, the comparison metric in Task 5 is comparing two different things and the recommendation is wrong. The lesson here is exam-relevant: when a team owns two pipelines that hit the same Feature Store, schema parity is what keeps downstream models consistent.

Build it in this order:

1. Install pyarrow if it is not already available (Colab and SageMaker Studio both ship it; bare Python environments may not).
2. Download one Parquet file from `output/glue/` and one from `output/wrangler/` into memory.
3. Read each with `pyarrow.parquet.read_table`. Compare column names (as sets) and row counts (within 1 percent tolerance).

Drift beyond 1 percent points at a one-hot bug: most often a categorical that one pipeline saw an extra category for, or a different `drop_first` default. The Task 5 cost comparison only makes sense if the two outputs are actually equivalent feature tables.

In [ ]:
# NBVAL_SKIP
# Task 4: Download one Parquet from each output prefix and compare
# schemas and row counts.

try:
    import pyarrow  # noqa: F401
    import pyarrow.parquet as pq
except ImportError:
    print("pyarrow not found; installing.")
    import subprocess as _sp
    _sp.check_call(["pip", "install", "--quiet", "pyarrow"])
    import pyarrow  # noqa: F401
    import pyarrow.parquet as pq

def first_parquet_key(prefix: str) -> str:
    listing = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    for obj in listing.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            return obj["Key"]
    raise RuntimeError(f"No parquet under {prefix}")

# YOUR CODE: Get the first parquet key from each prefix using
# first_parquet_key(GLUE_OUTPUT_PREFIX) and first_parquet_key(WRANGLER_OUTPUT_PREFIX).
# Assign to glue_key and wrangler_key.

glue_body = s3.get_object(Bucket=BUCKET_NAME, Key=glue_key)["Body"].read()
wrangler_body = s3.get_object(Bucket=BUCKET_NAME, Key=wrangler_key)["Body"].read()
print(f"Downloaded {len(glue_body) / (1024 * 1024):.2f} MB Glue parquet and "
      f"{len(wrangler_body) / (1024 * 1024):.2f} MB Wrangler-equivalent parquet.")

glue_tbl = pq.read_table(io.BytesIO(glue_body))
wrangler_tbl = pq.read_table(io.BytesIO(wrangler_body))

glue_cols = set(glue_tbl.column_names)
wrangler_cols = set(wrangler_tbl.column_names)
print(f"Glue columns: {sorted(glue_cols)}")
print(f"Wrangler-equivalent columns: {sorted(wrangler_cols)}")

glue_rows = glue_tbl.num_rows
wrangler_rows = wrangler_tbl.num_rows
print(f"Glue rows: {glue_rows:,}; Wrangler-equivalent rows: {wrangler_rows:,}")

common = glue_cols & wrangler_cols
missing_in_wrangler = glue_cols - wrangler_cols
missing_in_glue = wrangler_cols - glue_cols
print(f"Common columns: {len(common)}; only in Glue: {sorted(missing_in_wrangler)}; "
      f"only in Wrangler: {sorted(missing_in_glue)}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Re-download one parquet from each prefix, parse both,
# confirm matching column sets and row counts within 1 percent.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        try:
            import pyarrow.parquet as pq
        except ImportError:
            return CheckpointResult(
                status="error",
                error_reason="pyarrow is not installed. Re-run the Task 4 cell.",
            )

        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        def _first_parquet(prefix):
            listing = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
            for obj in listing.get("Contents", []):
                if obj["Key"].endswith(".parquet"):
                    return obj["Key"]
            return None

        g_key = _first_parquet(GLUE_OUTPUT_PREFIX)
        w_key = _first_parquet(WRANGLER_OUTPUT_PREFIX)
        if not g_key:
            return CheckpointResult(
                status="fail",
                error_reason=f"No .parquet under s3://{BUCKET_NAME}/{GLUE_OUTPUT_PREFIX}.",
            )
        if not w_key:
            return CheckpointResult(
                status="fail",
                error_reason=f"No .parquet under s3://{BUCKET_NAME}/{WRANGLER_OUTPUT_PREFIX}.",
            )

        g_body = s3_client.get_object(Bucket=BUCKET_NAME, Key=g_key)["Body"].read()
        w_body = s3_client.get_object(Bucket=BUCKET_NAME, Key=w_key)["Body"].read()
        g_tbl = pq.read_table(io.BytesIO(g_body))
        w_tbl = pq.read_table(io.BytesIO(w_body))

        g_cols = set(g_tbl.column_names)
        w_cols = set(w_tbl.column_names)
        if g_cols != w_cols:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Column sets differ. Only in Glue: {sorted(g_cols - w_cols)!r}. "
                    f"Only in Wrangler: {sorted(w_cols - g_cols)!r}. Most likely "
                    f"culprit: different drop_first defaults on one-hot encoding."
                ),
            )

        g_rows = g_tbl.num_rows
        w_rows = w_tbl.num_rows
        if g_rows == 0 or w_rows == 0:
            return CheckpointResult(
                status="fail",
                error_reason=f"Empty parquet. Glue rows: {g_rows}, Wrangler rows: {w_rows}.",
            )
        drift = abs(g_rows - w_rows) / max(g_rows, w_rows)
        if drift > 0.01:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Row count drift {drift * 100:.2f}% exceeds the 1% tolerance. "
                    f"Glue rows: {g_rows:,}; Wrangler rows: {w_rows:,}. The two "
                    f"pipelines are not producing equivalent outputs."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

One line is missing in the cell: pick the first parquet key from each prefix. The helper `first_parquet_key` already does the listing; you just need to call it twice and assign the results. The checkpoint message names which side failed.

</details>

<details><summary>Hint 2 (direction)</summary>

`first_parquet_key(GLUE_OUTPUT_PREFIX)` returns the Glue output key; `first_parquet_key(WRANGLER_OUTPUT_PREFIX)` returns the Wrangler-equivalent key. Both helpers walk the bucket's first listing page; with a single-output job each prefix holds exactly one parquet.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
glue_key = first_parquet_key(GLUE_OUTPUT_PREFIX)
wrangler_key = first_parquet_key(WRANGLER_OUTPUT_PREFIX)
```

</details>

**Wallet check.** Two S3 `GetObject` calls and an in-memory pyarrow parse. The list+get pair sits inside the always-free tier; the parse runs on your laptop. Damage so far: still about $0.01 cumulative. Your coffee was about $7. The ratio is unchanged.

## Task 5: Compute and print the comparison metric

Per blueprint Section 21, this is the surface that captures the comparative measurement. The checkpoint does not assert that one tool wins; it captures positive durations on both sides and prints the cost-per-run sentence. The reflection cell at the end of the notebook is where the comparative reasoning lives.

Build it in this order:

1. Compute Glue cost: `glue_dpu_seconds / 3600 * 0.44` per DPU-hour ($0.44 per DPU-hour at the 0.0625-DPU Python shell rate).
2. Compute Wrangler-equivalent cost: `wrangler_wall_clock_seconds / 3600 * 0.115` per instance-hour ($0.115 per hour for `ml.m5.large`).
3. Print the comparison line in the format the Option D card will surface verbatim: `Glue: X DPU-seconds at $0.44/DPU-hr = $Y; Wrangler-equivalent Processing: Z instance-seconds at $0.115/hr on ml.m5.large = $W.`

The numbers are real; they come straight from `glue.get_job_run` and `sagemaker.describe_processing_job`. The 5 GB-daily projection in the reflection MCQ uses these numbers as the per-run baseline.

In [ ]:
# NBVAL_SKIP
# Task 5: Compute and print the cost-per-run for both architectures.

GLUE_RATE_PER_DPU_HOUR = 0.44
WRANGLER_RATE_PER_HOUR = 0.115  # ml.m5.large on-demand, us-east-1.

# YOUR CODE: Compute glue_cost = glue_dpu_seconds / 3600 * GLUE_RATE_PER_DPU_HOUR.

# YOUR CODE: Compute wrangler_cost =
# wrangler_wall_clock_seconds / 3600 * WRANGLER_RATE_PER_HOUR.

comparison_line = (
    f"Glue: {glue_dpu_seconds:.1f} DPU-seconds at $0.44/DPU-hr = ${glue_cost:.4f}; "
    f"Wrangler-equivalent Processing: {wrangler_wall_clock_seconds:.1f} instance-seconds "
    f"at $0.115/hr on {wrangler_instance_type} = ${wrangler_cost:.4f}."
)
print(comparison_line)

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: glue_dpu_seconds and wrangler_wall_clock_seconds are both
# positive floats. Per blueprint Section 21 the validator does NOT contain
# comparison logic; it captures the measurement and surfaces it as
# printed output. Comparative reasoning lives in the reflection MCQs.

def checkpoint_5(session):
    from labrun_checks import CheckpointResult
    try:
        if not isinstance(glue_dpu_seconds, (int, float)) or glue_dpu_seconds <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"glue_dpu_seconds is {glue_dpu_seconds!r}. Re-run Task 2 so the "
                    f"DPU-seconds poll captures a positive value."
                ),
            )
        if (
            not isinstance(wrangler_wall_clock_seconds, (int, float))
            or wrangler_wall_clock_seconds <= 0
        ):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"wrangler_wall_clock_seconds is {wrangler_wall_clock_seconds!r}. "
                    f"Re-run Task 3 so the ProcessingStartTime/EndTime capture lands."
                ),
            )

        local_glue_cost = glue_dpu_seconds / 3600 * 0.44
        local_wrangler_cost = wrangler_wall_clock_seconds / 3600 * 0.115
        print(
            f"Glue: {glue_dpu_seconds:.1f} DPU-seconds at $0.44/DPU-hr = ${local_glue_cost:.4f}; "
            f"Wrangler-equivalent Processing: {wrangler_wall_clock_seconds:.1f} "
            f"instance-seconds at $0.115/hr on {wrangler_instance_type} = "
            f"${local_wrangler_cost:.4f}."
        )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Two arithmetic lines are missing. Each one divides a seconds count by 3600 to get hours, then multiplies by the rate per hour. The variables already exist; you only need the math.

</details>

<details><summary>Hint 2 (direction)</summary>

The Glue rate is `$0.44 per DPU-hour` so `glue_cost = glue_dpu_seconds / 3600 * 0.44`. The Wrangler-equivalent rate is `$0.115 per instance-hour` so `wrangler_cost = wrangler_wall_clock_seconds / 3600 * 0.115`. The print line uses both variables; do not rename them.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 5:

```python
glue_cost = glue_dpu_seconds / 3600 * GLUE_RATE_PER_DPU_HOUR
wrangler_cost = wrangler_wall_clock_seconds / 3600 * WRANGLER_RATE_PER_HOUR
```

</details>

**Wallet check.** Pure arithmetic on Python floats; no AWS calls. Damage so far: still about $0.01 cumulative. The comparison line above is the artifact the lead wanted; everything else is plumbing.

## Cleanup

Both pipelines bill only while running. The Glue job and Processing job both finished, so nothing is accruing cost right now. Cleanup still tears everything down to keep the orphan scan clean for next time. If the Processing job is somehow still running when you hit this cell, the manual `stop_processing_job` call fires first; otherwise it is a no-op.

Re-running the cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Stop any in-flight Processing job manually, then run_cleanup
# walks the manifest in reverse-creation order (glue_job, iam_role x2,
# s3_bucket). The s3_bucket adapter empties the first listing page before
# deleting the bucket; this lab creates at most ~5-10 objects so the
# single-page limit (1000) is not a concern.

import sys

manual_warnings = []

if WRANGLER_JOB_NAME:
    sm_cleanup = boto3.client(
        "sagemaker",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    try:
        desc = sm_cleanup.describe_processing_job(ProcessingJobName=WRANGLER_JOB_NAME)
        if desc["ProcessingJobStatus"] == "InProgress":
            sm_cleanup.stop_processing_job(ProcessingJobName=WRANGLER_JOB_NAME)
            print(f"Stopped in-flight Processing job: {WRANGLER_JOB_NAME}")
    except ClientError as e:
        if e.response["Error"]["Code"] not in (
            "ValidationException", "ResourceNotFound",
        ):
            manual_warnings.append(
                f"FAILED TO STOP: processing_job {WRANGLER_JOB_NAME!r}. Error: {e}. "
                f"Run manually: aws sagemaker stop-processing-job "
                f"--processing-job-name {WRANGLER_JOB_NAME}"
            )

result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if manual_warnings or result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
critical_destroyed = 0
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (failed_count == 0 and result.status == "clean") else "dirty"
cleanup(status=final_status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.01 to $0.02.** Glue Python shell at 0.0625 DPU for one run was a fraction of a cent. SageMaker Processing on `ml.m5.large` for one run was about a cent. S3 PUT and GET at this scale sat inside the always-free tier. IAM, STS, and tag operations were free. Check your AWS Billing console in 24 hours to confirm zero ongoing charges.

## Reflection

These are not graded. They are for you.

1. Walk through the trade-off matrix between Glue Spark ETL, Glue Python shell, SageMaker Data Wrangler, SageMaker Processing, and EMR Spark for a 5 GB daily feature-prep job. List the per-run cost, the developer skill needed, the maximum data volume each one comfortably scales to, and the production-readiness story for each. For the scenario in this lab (5 GB daily, three analysts, two engineers), which would you pick and why?

2. Data Wrangler exports flows as `.flow` JSON files that get baked into a SageMaker Processing Job. Walk through what that means for source control, code review, and reproducibility compared to a Python or Spark script. Where does the Data Wrangler GUI surface help, and where does it cost you?

## Exam-Style Practice

**Question 1 (Domain 1, choosing between Glue ETL and Data Wrangler; constraint-driven comparison per blueprint Section 21):**

A platform team is choosing the feature-prep tool for a churn-modeling pipeline. The pipeline runs daily against 5 GB of customer data, the team has three analysts who do not write Python and two engineers who write Spark, and the pipeline output feeds SageMaker Feature Store. Which tool fits the team and workload with the least operational overhead?

A. SageMaker Data Wrangler. Build the flow in the Studio GUI, schedule it as a Processing Job, write to Feature Store via the built-in Feature Store output node.

B. Glue Spark ETL on G.1X DPU. Write a Spark script, schedule a daily run via EventBridge, write to Feature Store via boto3.

C. SageMaker Processing with a custom container the team builds in-house, scheduled via EventBridge.

D. EMR Spark cluster running daily for 2 hours, scheduled via Step Functions.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: Data Wrangler is the AWS-recommended low-code feature-prep tool, built specifically for analysts who do not write Python or Spark. The built-in Feature Store output node eliminates the engineering integration. At 5 GB daily, Data Wrangler on `ml.m5.4xlarge` runs in 5-10 minutes per day; the operational overhead is low.
- B is wrong on operational overhead for the team: Glue Spark requires Spark expertise the three analysts do not have. Two engineers can write it but the resulting pipeline is opaque to the analysts who own the feature logic.
- C is wrong: building and maintaining a custom Processing container is the highest-overhead option. Data Wrangler exists specifically to avoid this.
- D is wrong: EMR Spark is overkill for 5 GB daily. EMR is the right tool at the 1 TB+ daily scale; at 5 GB it adds cluster-lifecycle ops with no throughput benefit.

</details>

**Question 2 (Domain 1, when to switch tools at scale):**

A pipeline that originally ran a 50 MB feature-prep job in SageMaker Data Wrangler now ingests 800 GB per day after a product launch. The team is hitting `OutOfMemory` errors on `ml.m5.4xlarge` and the per-run cost has climbed past $20/day. Which migration path is the AWS-recommended fit?

A. Increase the Data Wrangler Processing Job instance type to `ml.m5.24xlarge` and rely on vertical scaling.

B. Export the Data Wrangler flow as a notebook, then port the logic to a Glue Spark ETL job with autoscaling DPUs.

C. Rewrite the flow as a Lambda function with the 15-minute timeout extension.

D. Move the data to DynamoDB and use DynamoDB Streams to trigger a Step Functions feature-prep workflow.

<details><summary>Show answer</summary>

**Correct: B.**

- A is partially right that vertical scaling buys time but it tops out around `ml.m5.24xlarge` and the per-hour cost climbs fast. At 800 GB daily Data Wrangler is the wrong tool regardless of instance size.
- B is correct: Data Wrangler exports flows as Spark or pandas notebooks via the "Export to..." menu. The AWS-recommended migration path for a flow that has outgrown Data Wrangler is to export the equivalent Spark script and run it on Glue ETL with autoscaling, where the per-DPU-minute cost is much lower than Data Wrangler at the same throughput.
- C is wrong: 800 GB per day exceeds Lambda's processing capacity by orders of magnitude. Lambda has a 10 GB memory ceiling and a 15-minute timeout.
- D is wrong: DynamoDB is a key-value store optimized for low-latency reads, not for batch feature-prep on 800 GB of data. Step Functions is an orchestrator, not a data processor.

</details>